In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv(
    "../data/raw/DNRE2024.csv",
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

# Remover registros sem peso
df = df.dropna(subset=["PESO"]).copy()

# Criar alvo
df["baixo_peso"] = (df["PESO"] < 2500).astype(int)

print("Registros válidos:", len(df))

Registros válidos: 120785


In [27]:
# Selecionar colunas candidatas e renomear para nomes mais amigáveis
mapa_colunas = {
    "FxEtPartur": "faixa_etaria_mae",
    "RACACORMAE": "raca_cor_mae",
    "ESTCIVMAE": "estado_civil_mae",
    "ESCMAE2010": "escolaridade_mae",
    "QTDGESTANT": "qtd_gestacoes",
    "QTDPARTNOR": "qtd_partos_normais",
    "QTDPARTCES": "qtd_partos_cesareas",
    "QTDFILVIVO": "qtd_filhos_vivos",
    "QTDFILMORT": "qtd_filhos_mortos",
    "CONSPRENAT": "qtd_consultas_prenatal",
    "MESPRENAT": "mes_inicio_prenatal",
    "GRAVIDEZ": "tipo_gravidez"
}

X = df[list(mapa_colunas.keys())].rename(
    columns=mapa_colunas
).copy()

y = df["baixo_peso"].copy()

# Tratar códigos especiais
X["qtd_consultas_prenatal"] = (
    X["qtd_consultas_prenatal"].replace(99, np.nan)
)

X["mes_inicio_prenatal"] = (
    X["mes_inicio_prenatal"]
    .where(X["mes_inicio_prenatal"].between(1, 10))
)

categoricas_codificadas = [
    "raca_cor_mae",
    "estado_civil_mae",
    "escolaridade_mae",
    "tipo_gravidez"
]

for coluna in categoricas_codificadas:
    X[coluna] = X[coluna].replace(9, np.nan)
    X[coluna] = X[coluna].astype("string").fillna("ignorado")

X["faixa_etaria_mae"] = (
    X["faixa_etaria_mae"]
    .replace("Ign", "ignorado")
    .fillna("ignorado")
    .astype(str)
)

# Verificar se o peso não está entre os preditores
assert "PESO" not in X.columns

print("Dimensão de X:", X.shape)
print("Dimensão de y:", y.shape)

display(X.head())

Dimensão de X: (120785, 12)
Dimensão de y: (120785,)


,faixa_etaria_mae,raca_cor_mae,estado_civil_mae,escolaridade_mae,qtd_gestacoes,qtd_partos_normais,qtd_partos_cesareas,qtd_filhos_vivos,qtd_filhos_mortos,qtd_consultas_prenatal,mes_inicio_prenatal,tipo_gravidez
0,25–29,2.0,1.0,2.0,2.0,1.0,0.0,1.0,1.0,8.0,1.0,1.0
1,30–34,4.0,1.0,5.0,2.0,2.0,0.0,2.0,0.0,17.0,1.0,1.0
2,20–24,4.0,2.0,3.0,2.0,0.0,0.0,0.0,2.0,9.0,1.0,1.0
3,30–34,4.0,2.0,5.0,1.0,0.0,1.0,0.0,1.0,NaN,1.0,1.0
4,25–29,ignorado,1.0,5.0,1.0,0.0,1.0,1.0,0.0,5.0,3.0,1.0


In [28]:
# Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Treinamento:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição no treinamento:")
print(y_train.value_counts(normalize=True).round(4))

print("\nDistribuição no teste:")
print(y_test.value_counts(normalize=True).round(4))

Treinamento: (96628, 12)
Teste: (24157, 12)

Distribuição no treinamento:
baixo_peso
0    0.8967
1    0.1033
Name: proportion, dtype: float64

Distribuição no teste:
baixo_peso
0    0.8968
1    0.1032
Name: proportion, dtype: float64


In [29]:
# Pré - processamento para modelagem
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

In [30]:
colunas_numericas = [
    "qtd_gestacoes",
    "qtd_partos_normais",
    "qtd_partos_cesareas",
    "qtd_filhos_vivos",
    "qtd_filhos_mortos",
    "qtd_consultas_prenatal",
    "mes_inicio_prenatal"
]

colunas_categoricas = [
    "faixa_etaria_mae",
    "raca_cor_mae",
    "estado_civil_mae",
    "escolaridade_mae",
    "tipo_gravidez"
]

print("Numéricas:", len(colunas_numericas))
print("Categóricas:", len(colunas_categoricas))

Numéricas: 7
Categóricas: 5


In [31]:
# Contrução do pipeline de pré-processamento:
# Mediana: para preencher valores numéricos ausentes.
# Indicadores de ausência: para registrar quais campos numéricos estavam faltando.
# One-Hot Encoding: para transformar categorias em colunas numéricas.
# StandardScaler: para padronizar as variáveis numéricas.
pipeline_numerico = Pipeline([
    ("imputador", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("normalizador", StandardScaler())
])

pipeline_categorico = Pipeline([
    ("imputador", SimpleImputer(
        strategy="constant",
        fill_value="ignorado"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessador = ColumnTransformer([
    ("numericas", pipeline_numerico, colunas_numericas),
    ("categoricas", pipeline_categorico, colunas_categoricas)
])

print("Pré-processamento configurado!")

Pré-processamento configurado!


In [32]:
# Dois primeiros modelos: referência e regressão logística
from sklearn.base import clone

modelos = {
    "Dummy": DummyClassifier(
        strategy="prior"
    ),

    "Regressão Logística": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )
}

pipelines = {
    nome: Pipeline([
        ("preprocessamento", clone(preprocessador)),
        ("classificador", modelo)
    ])
    for nome, modelo in modelos.items()
}

print("Modelos configurados:")
print(list(pipelines.keys()))

Modelos configurados:
['Dummy', 'Regressão Logística']


In [33]:
from sklearn.metrics import make_scorer, fbeta_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

metricas = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "recall": "recall",
    "precision": "precision",
    "f2": make_scorer(fbeta_score, beta=2, zero_division=0)
}

print("Métricas configuradas com sucesso!")

Métricas configuradas com sucesso!


In [34]:
# Comparação
resultados = []

for nome, pipeline in pipelines.items():

    print(f"\nTreinando: {nome}")

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=metricas,
        n_jobs=-1
    )

    resultado = {"modelo": nome}

    for metrica in metricas:
        valores = scores[f"test_{metrica}"]

        resultado[f"{metrica}_media"] = valores.mean()
        resultado[f"{metrica}_desvio"] = valores.std()

    resultados.append(resultado)

tabela_resultados = pd.DataFrame(resultados)

display(tabela_resultados.round(4))


Treinando: Dummy

Treinando: Regressão Logística


,modelo,pr_auc_media,pr_auc_desvio,roc_auc_media,roc_auc_desvio,recall_media,recall_desvio,precision_media,precision_desvio,f2_media,f2_desvio
0,Dummy,0.1033,0.0000,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,Regressão Logística,0.3467,0.0058,0.7306,0.0016,0.6091,0.0073,0.2143,0.0022,0.4451,0.0043


- **PR-AUC de 0,3467:** a Regressão Logística apresenta capacidade de classificação e ordenação dos casos positivos superior ao modelo de referência, cuja PR-AUC é 0,1033.



- **ROC-AUC de 0,7306:** o modelo apresenta capacidade moderada de distinguir registros com e sem baixo peso.



- **Recall de 60,91%:** com o limiar padrão de 0,5, identifica aproximadamente 61 de cada 100 casos de baixo peso.



- **Precision de 21,43%:** entre os registros classificados como baixo peso, aproximadamente 21 de cada 100 realmente pertencem a essa classe.



- **F2-score de 0,4451:** resume o equilíbrio entre Precision e Recall, dando maior importância ao Recall.

In [35]:
# Random Forest e HistGradientBoosting
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)

novos_modelos = {
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state=42
    )
}

for nome, modelo in novos_modelos.items():
    pipelines[nome] = Pipeline([
        ("preprocessamento", clone(preprocessador)),
        ("classificador", modelo)
    ])

print("Modelos disponíveis:")
print(list(pipelines.keys()))

Modelos disponíveis:
['Dummy', 'Regressão Logística', 'Random Forest', 'HistGradientBoosting']


In [36]:
for nome in ["Random Forest", "HistGradientBoosting"]:

    print(f"\nTreinando: {nome}")

    scores = cross_validate(
        pipelines[nome],
        X_train,
        y_train,
        cv=cv,
        scoring=metricas,
        n_jobs=1
    )

    resultado = {"modelo": nome}

    for metrica in metricas:
        valores = scores[f"test_{metrica}"]

        resultado[f"{metrica}_media"] = valores.mean()
        resultado[f"{metrica}_desvio"] = valores.std()

    resultados.append(resultado)

    print(f"{nome} finalizado!")

tabela_resultados = pd.DataFrame(resultados)

tabela_resultados = tabela_resultados.sort_values(
    "pr_auc_media",
    ascending=False
)

display(tabela_resultados.round(4))


Treinando: Random Forest
Random Forest finalizado!

Treinando: HistGradientBoosting
HistGradientBoosting finalizado!


,modelo,pr_auc_media,pr_auc_desvio,roc_auc_media,roc_auc_desvio,recall_media,recall_desvio,precision_media,precision_desvio,f2_media,f2_desvio
3,HistGradientBoosting,0.3588,0.0048,0.7371,0.0020,0.5678,0.0120,0.2349,0.0019,0.4423,0.0054
2,Random Forest,0.3477,0.0051,0.7337,0.0035,0.5301,0.0059,0.2525,0.0023,0.4345,0.0031
1,Regressão Logística,0.3467,0.0058,0.7306,0.0016,0.6091,0.0073,0.2143,0.0022,0.4451,0.0043
0,Dummy,0.1033,0.0000,0.5000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [37]:
# Salvar os resultados
from pathlib import Path

pasta_resultados = Path("../docs")
pasta_resultados.mkdir(exist_ok=True)

tabela_resultados.to_csv(
    pasta_resultados / "resultados_cv.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resultados salvos em docs/resultados_cv.csv")

Resultados salvos em docs/resultados_cv.csv


O HistGradientBoosting apresentou a maior PR-AUC, com 0,3588.

Entretanto, a Regressão Logística apresentou o maior Recall e F2-score usando o limiar padrão de 0,5.

In [38]:
# Testar o modelo sem raça/cor materna
# Criar conjunto sem raça/cor materna
X_train_sem_raca = X_train.drop(
    columns=["raca_cor_mae"]
)

# Atualizar as colunas categóricas
categoricas_sem_raca = [
    coluna
    for coluna in colunas_categoricas
    if coluna != "raca_cor_mae"
]

# Criar pré-processador sem raça/cor
preprocessador_sem_raca = ColumnTransformer([
    ("numericas", clone(pipeline_numerico), colunas_numericas),
    ("categoricas", clone(pipeline_categorico), categoricas_sem_raca)
])

print("Variáveis restantes:", X_train_sem_raca.shape[1])

Variáveis restantes: 11


In [39]:
# Modelo sem raça/cor materna
modelo_sem_raca = Pipeline([
    ("preprocessamento", preprocessador_sem_raca),
    ("classificador", HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

# Avaliar com validação cruzada
scores_sem_raca = cross_validate(
    modelo_sem_raca,
    X_train_sem_raca,
    y_train,
    cv=cv,
    scoring=metricas,
    n_jobs=1
)

# Organizar comparação
comparacao_raca = pd.DataFrame({
    "Modelo": [
        "Com raça/cor",
        "Sem raça/cor"
    ],
    "PR-AUC": [
        tabela_resultados.loc[
            tabela_resultados["modelo"] == "HistGradientBoosting",
            "pr_auc_media"
        ].iloc[0],
        scores_sem_raca["test_pr_auc"].mean()
    ],
    "Recall": [
        tabela_resultados.loc[
            tabela_resultados["modelo"] == "HistGradientBoosting",
            "recall_media"
        ].iloc[0],
        scores_sem_raca["test_recall"].mean()
    ],
    "F2": [
        tabela_resultados.loc[
            tabela_resultados["modelo"] == "HistGradientBoosting",
            "f2_media"
        ].iloc[0],
        scores_sem_raca["test_f2"].mean()
    ]
})

display(comparacao_raca.round(4))

,Modelo,PR-AUC,Recall,F2
0,Com raça/cor,0.3588,0.5678,0.4423
1,Sem raça/cor,0.3593,0.5667,0.4411


### Experimento de remoção da variável raça/cor materna

Foi realizado um experimento comparativo com o algoritmo HistGradientBoosting, utilizando validação cruzada estratificada de cinco divisões, com e sem a variável raça/cor materna.

O modelo com a variável apresentou PR-AUC de 0,3588, enquanto o modelo sem a variável alcançou 0,3593. O Recall apresentou uma pequena redução de 0,5678 para 0,5667, e o F2-score passou de 0,4423 para 0,4411.

Considerando que a remoção não prejudicou a PR-AUC e que raça/cor materna representa um atributo sensível, optou-se por manter a versão sem essa variável para os próximos experimentos. A decisão será reavaliada caso novos resultados indiquem necessidade.

In [40]:
tabela_resultados.to_csv(
    "../docs/resultados_cv.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resultados salvos com sucesso!")

Resultados salvos com sucesso!


In [41]:
# testar novas variáveis derivadas
# total_partos_anteriores: soma dos partos normais e cesáreos anteriores.
# inicio_prenatal_tardio: indica início do pré-natal a partir do 4º mês.
from sklearn.preprocessing import FunctionTransformer

def criar_variaveis_derivadas(X):
    X = X.copy()

    X["total_partos_anteriores"] = (
        X["qtd_partos_normais"] +
        X["qtd_partos_cesareas"]
    )

    X["inicio_prenatal_tardio"] = np.where(
        X["mes_inicio_prenatal"].isna(),
        np.nan,
        (X["mes_inicio_prenatal"] >= 4).astype(int)
    )

    return X

transformador_features = FunctionTransformer(
    criar_variaveis_derivadas,
    validate=False
)

print("Transformador configurado!")

Transformador configurado!


In [42]:
numericas_derivadas = colunas_numericas + [
    "total_partos_anteriores",
    "inicio_prenatal_tardio"
]

preprocessador_derivado = ColumnTransformer([
    ("numericas", clone(pipeline_numerico), numericas_derivadas),
    ("categoricas", clone(pipeline_categorico), categoricas_sem_raca)
])

modelo_com_derivadas = Pipeline([
    ("criacao_features", transformador_features),
    ("preprocessamento", preprocessador_derivado),
    ("classificador", HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

print("Pipeline com variáveis derivadas configurado!")

Pipeline com variáveis derivadas configurado!


In [43]:
# Avaliação e comparação
scores_derivadas = cross_validate(
    modelo_com_derivadas,
    X_train_sem_raca,
    y_train,
    cv=cv,
    scoring=metricas,
    n_jobs=1
)

comparacao_features = pd.DataFrame({
    "Modelo": [
        "Sem variáveis derivadas",
        "Com variáveis derivadas"
    ],
    "PR-AUC": [
        scores_sem_raca["test_pr_auc"].mean(),
        scores_derivadas["test_pr_auc"].mean()
    ],
    "Recall": [
        scores_sem_raca["test_recall"].mean(),
        scores_derivadas["test_recall"].mean()
    ],
    "F2": [
        scores_sem_raca["test_f2"].mean(),
        scores_derivadas["test_f2"].mean()
    ]
})

display(comparacao_features.round(4))

,Modelo,PR-AUC,Recall,F2
0,Sem variáveis derivadas,0.3593,0.5667,0.4411
1,Com variáveis derivadas,0.3592,0.5650,0.4403


### Experimento com variáveis derivadas

Foi realizado um experimento com o algoritmo HistGradientBoosting para avaliar a inclusão das variáveis `total_partos_anteriores` e `inicio_prenatal_tardio`.

O modelo sem as variáveis derivadas apresentou PR-AUC de 0,3593, Recall de 0,5667 e F2-score de 0,4411.

Com a inclusão das novas variáveis, os resultados foram PR-AUC de 0,3592, Recall de 0,5650 e F2-score de 0,4403.

Como não houve melhoria de desempenho, optou-se por manter a configuração mais simples, sem essas variáveis adicionais.

In [44]:
# indicadores de faixa etária materna
# verificar se indicadores específicos para gestação na adolescência e idade materna avançada melhoram o desempenho.
def criar_indicadores_idade(X):
    X = X.copy()

    X["mae_adolescente"] = (
        X["faixa_etaria_mae"]
        .isin(["10–14", "15–19"])
        .astype(int)
    )

    X["idade_materna_avancada"] = (
        X["faixa_etaria_mae"]
        .isin(["35–39", "40–44", "45–49", ">=50"])
        .astype(int)
    )

    return X

transformador_idade = FunctionTransformer(
    criar_indicadores_idade,
    validate=False
)

print("Indicadores de idade configurados!")

Indicadores de idade configurados!


In [45]:
# Configuração do pipeline
numericas_idade = colunas_numericas + [
    "mae_adolescente",
    "idade_materna_avancada"
]

preprocessador_idade = ColumnTransformer([
    ("numericas", clone(pipeline_numerico), numericas_idade),
    ("categoricas", clone(pipeline_categorico), categoricas_sem_raca)
])

modelo_com_idade = Pipeline([
    ("criacao_features", transformador_idade),
    ("preprocessamento", preprocessador_idade),
    ("classificador", HistGradientBoostingClassifier(
        class_weight="balanced",
        random_state=42
    ))
])

print("Pipeline com indicadores de idade configurado!")

Pipeline com indicadores de idade configurado!


In [46]:
# Avaliar os indicadores
scores_idade = cross_validate(
    modelo_com_idade,
    X_train_sem_raca,
    y_train,
    cv=cv,
    scoring=metricas,
    n_jobs=1
)

comparacao_idade = pd.DataFrame({
    "Modelo": [
        "Sem indicadores de idade",
        "Com indicadores de idade"
    ],
    "PR-AUC": [
        scores_sem_raca["test_pr_auc"].mean(),
        scores_idade["test_pr_auc"].mean()
    ],
    "Recall": [
        scores_sem_raca["test_recall"].mean(),
        scores_idade["test_recall"].mean()
    ],
    "F2": [
        scores_sem_raca["test_f2"].mean(),
        scores_idade["test_f2"].mean()
    ]
})

display(comparacao_idade.round(4))

,Modelo,PR-AUC,Recall,F2
0,Sem indicadores de idade,0.3593,0.5667,0.4411
1,Com indicadores de idade,0.3588,0.5694,0.4424


### Seleção da configuração candidata do modelo

Após a comparação dos algoritmos DummyClassifier, Regressão Logística, Random Forest e HistGradientBoosting, o HistGradientBoosting apresentou a maior PR-AUC média na validação cruzada.

Foram realizados experimentos adicionais para avaliar a remoção da variável raça/cor materna e a inclusão de variáveis derivadas relacionadas ao histórico de partos, início do pré-natal e faixa etária materna.

A configuração sem raça/cor e sem variáveis derivadas apresentou a maior PR-AUC média, de 0,3593, com Recall de 0,5667 e F2-score de 0,4411.

In [47]:
# Gerar as probabilidades OOF
from sklearn.model_selection import cross_val_predict

# Gerar probabilidades usando somente os dados de treinamento
probas_oof = cross_val_predict(
    modelo_sem_raca,
    X_train_sem_raca,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=1
)[:, 1]

print("Probabilidades geradas:", len(probas_oof))
print("Mínima:", probas_oof.min())
print("Máxima:", probas_oof.max())

Probabilidades geradas: 96628
Mínima: 0.03917504414029365
Máxima: 0.9877662466315027


In [48]:
# Salvar o arquivo
from pathlib import Path

pasta_processed = Path("../data/processed")
pasta_processed.mkdir(parents=True, exist_ok=True)

oof_df = pd.DataFrame({
    "y_true": y_train.to_numpy(),
    "proba": probas_oof
})

oof_df.to_csv(
    pasta_processed / "oof_proba.csv",
    index=False,
    encoding="utf-8"
)

print("Arquivo salvo em data/processed/oof_proba.csv")

display(oof_df.head())

Arquivo salvo em data/processed/oof_proba.csv


,y_true,proba
0,0,0.443185
1,0,0.391556
2,0,0.421276
3,1,0.632429
4,0,0.445886


In [49]:
print("Quantidade de registros:", len(oof_df))

print("\nValores ausentes:")
print(oof_df.isna().sum())

print("\nEstatísticas das probabilidades:")
display(oof_df["proba"].describe())

Quantidade de registros: 96628

Valores ausentes:
y_true    0
proba     0
dtype: int64

Estatísticas das probabilidades:


count    96628.000000
mean         0.420645
std          0.170980
min          0.039175
25%          0.292735
50%          0.381513
75%          0.500149
max          0.987766
Name: proba, dtype: float64

In [50]:
# Treinamento do modelo candidato
from sklearn.base import clone

# Criar uma cópia independente do pipeline selecionado
modelo_final = clone(modelo_sem_raca)

# Treinar com todos os registros de treinamento
modelo_final.fit(
    X_train_sem_raca,
    y_train
)

print("Modelo treinado com sucesso!")
print("Registros utilizados:", len(X_train_sem_raca))

Modelo treinado com sucesso!
Registros utilizados: 96628


In [52]:
# Salvar o modelo treinado
import joblib
from pathlib import Path

pasta_modelos = Path("../models")
pasta_modelos.mkdir(parents=True, exist_ok=True)

caminho_modelo = (
    pasta_modelos / "modelo_baixo_peso_v1.0.0.joblib"
)

joblib.dump(
    modelo_final,
    caminho_modelo,
    compress=3
)

print("Modelo salvo em:", caminho_modelo)

Modelo salvo em: ..\models\modelo_baixo_peso_v1.0.0.joblib


In [53]:
modelo_carregado = joblib.load(caminho_modelo)

# Selecionar um exemplo do treinamento
exemplo = X_train_sem_raca.iloc[[0]]

# Gerar previsão
probabilidade = modelo_carregado.predict_proba(
    exemplo
)[0, 1]

print("Modelo carregado com sucesso!")
print("Saída do modelo:", round(probabilidade, 4))

Modelo carregado com sucesso!
Saída do modelo: 0.4282


In [54]:
# 11 variáveis que o modelo espera receber.(backend)
import json

schema = {
    "versao": "1.0.0",
    "variaveis": {
        "faixa_etaria_mae": "categoria",
        "estado_civil_mae": "categoria",
        "escolaridade_mae": "categoria",
        "tipo_gravidez": "categoria",
        "qtd_gestacoes": "numero",
        "qtd_partos_normais": "numero",
        "qtd_partos_cesareas": "numero",
        "qtd_filhos_vivos": "numero",
        "qtd_filhos_mortos": "numero",
        "qtd_consultas_prenatal": "numero",
        "mes_inicio_prenatal": "numero"
    }
}

with open(
    "../models/input_schema.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(schema, arquivo, indent=4, ensure_ascii=False)

print("input_schema.json criado com sucesso!")

input_schema.json criado com sucesso!


In [55]:
metadata = {
    "nome": "Modelo de previsão de baixo peso ao nascer",
    "versao": "1.0.0",
    "algoritmo": "HistGradientBoostingClassifier",
    "dataset": "SINASC 2024",
    "registros_treinamento": len(X_train_sem_raca),
    "quantidade_variaveis": X_train_sem_raca.shape[1],
    "metricas_validacao_cruzada": {
        "pr_auc": 0.3593,
        "recall": 0.5667,
        "f2_score": 0.4411
    },
    "limiar_classificacao": None,
    "calibracao": "pendente",
    "status": "candidato"
}

with open(
    "../models/metadata.json",
    "w",
    encoding="utf-8"
) as arquivo:
    json.dump(metadata, arquivo, indent=4, ensure_ascii=False)

print("metadata.json criado com sucesso!")

metadata.json criado com sucesso!


In [56]:
from pathlib import Path

for arquivo in Path("../models").iterdir():
    print(arquivo.name)

input_schema.json
metadata.json
modelo_baixo_peso_v1.0.0.joblib


In [57]:
import sys
from pathlib import Path

raiz = Path.cwd().resolve().parent

if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.ml.predict import predict_one

# Usar um registro real de treinamento como exemplo
dados_teste = X_train_sem_raca.iloc[0].to_dict()

resultado = predict_one(dados_teste)

print(resultado)

{'versao_modelo': '1.0.0', 'score_baixo_peso': 0.428166, 'classificacao': None, 'limiar': None, 'calibrado': False}


Esse teste confirma se o arquivo predict.py consegue localizar o modelo salvo, receber um registro com as 11 variáveis e retornar uma previsão no formato de dicionário.

In [58]:
# Testes
# TESTE 1: Campo obrigatório ausente

dados_incompletos = dados_teste.copy()
dados_incompletos.pop("qtd_gestacoes")

try:
    predict_one(dados_incompletos)
except ValueError as erro:
    print("TESTE 1 - OK:", erro)


# TESTE 2: Campo desconhecido

dados_extras = dados_teste.copy()
dados_extras["campo_inexistente"] = 123

try:
    predict_one(dados_extras)
except ValueError as erro:
    print("TESTE 2 - OK:", erro)


# TESTE 3: Valor numérico inválido

dados_invalidos = dados_teste.copy()
dados_invalidos["qtd_gestacoes"] = "abc"

try:
    predict_one(dados_invalidos)
except (ValueError, TypeError) as erro:
    print("TESTE 3 - OK:", erro)

TESTE 1 - OK: Campos obrigatórios ausentes: ['qtd_gestacoes']
TESTE 2 - OK: Campos desconhecidos: ['campo_inexistente']
TESTE 3 - OK: Unable to parse string 'abc' at position 0


In [59]:
import numpy as np

# TESTE 4: Valor numérico ausente
dados_ausentes = dados_teste.copy()
dados_ausentes["qtd_gestacoes"] = None

try:
    resultado = predict_one(dados_ausentes)
    print("TESTE 4 - OK:", resultado)
except Exception as erro:
    print("TESTE 4 - ERRO:", erro)


# TESTE 5: Categoria não vista no treinamento
dados_categoria_nova = dados_teste.copy()
dados_categoria_nova["estado_civil_mae"] = "categoria_nova"

try:
    resultado = predict_one(dados_categoria_nova)
    print("TESTE 5 - OK:", resultado)
except Exception as erro:
    print("TESTE 5 - ERRO:", erro)

TESTE 4 - OK: {'versao_modelo': '1.0.0', 'score_baixo_peso': 0.428166, 'classificacao': None, 'limiar': None, 'calibrado': False}
TESTE 5 - OK: {'versao_modelo': '1.0.0', 'score_baixo_peso': 0.428166, 'classificacao': None, 'limiar': None, 'calibrado': False}


In [60]:
# Verificar categorias 
for coluna in categoricas_sem_raca:
    print(f"\nCOLUNA: {coluna}")

    valores = (
        X_train_sem_raca[coluna]
        .value_counts(dropna=False)
        .sort_index()
    )

    print(valores.to_string())


COLUNA: faixa_etaria_mae
faixa_etaria_mae
10–14         199
15–19        6591
20–24       19835
25–29       24112
30–34       22386
35–39       17459
40–44        5594
45–49         416
>=50           35
ignorado        1

COLUNA: estado_civil_mae
estado_civil_mae
1.0         45018
2.0         33535
3.0           101
4.0          1681
5.0         16219
ignorado       74

COLUNA: escolaridade_mae
escolaridade_mae
0.0            36
1.0           581
2.0          8813
3.0         53152
4.0          5590
5.0         28435
ignorado       21

COLUNA: tipo_gravidez
tipo_gravidez
1.0         94004
2.0          2572
3.0            44
ignorado        8


In [61]:
# Verificar valores numericos
resumo_numerico = X_train_sem_raca[
    colunas_numericas
].describe().T

display(
    resumo_numerico[
        ["count", "min", "25%", "50%", "75%", "max"]
    ]
)

,count,min,25%,50%,75%,max
qtd_gestacoes,96611.0,0.0,0.0,1.0,2.0,24.0
qtd_partos_normais,96608.0,0.0,0.0,0.0,1.0,36.0
qtd_partos_cesareas,96613.0,0.0,0.0,0.0,0.0,30.0
qtd_filhos_vivos,96608.0,0.0,0.0,1.0,1.0,11.0
qtd_filhos_mortos,96594.0,0.0,0.0,0.0,0.0,20.0
qtd_consultas_prenatal,96576.0,0.0,8.0,10.0,12.0,40.0
mes_inicio_prenatal,95826.0,1.0,1.0,2.0,2.0,10.0


In [62]:
for coluna in categoricas_sem_raca:
    valores = (
        X_train_sem_raca[coluna]
        .astype(str)
        .unique()
        .tolist()
    )

    print(f"{coluna}: {sorted(valores)}")

faixa_etaria_mae: ['10–14', '15–19', '20–24', '25–29', '30–34', '35–39', '40–44', '45–49', '>=50', 'ignorado']
estado_civil_mae: ['1.0', '2.0', '3.0', '4.0', '5.0', 'ignorado']
escolaridade_mae: ['0.0', '1.0', '2.0', '3.0', '4.0', '5.0', 'ignorado']
tipo_gravidez: ['1.0', '2.0', '3.0', 'ignorado']


In [63]:
import json
from pathlib import Path

caminho_schema = Path("../models/input_schema.json")

with open(caminho_schema, encoding="utf-8") as arquivo:
    schema = json.load(arquivo)

# Registrar categorias observadas no treinamento
schema["categorias_observadas"] = {
    coluna: sorted(
        X_train_sem_raca[coluna]
        .astype(str)
        .unique()
        .tolist()
    )
    for coluna in categoricas_sem_raca
}

# Regras iniciais de validação
schema["regras_numericas"] = {
    coluna: {
        "minimo": 0,
        "inteiro": True,
        "permite_nulo": True
    }
    for coluna in colunas_numericas
}

# Regra específica para o mês do pré-natal
schema["regras_numericas"]["mes_inicio_prenatal"]["minimo"] = 1
schema["regras_numericas"]["mes_inicio_prenatal"]["maximo"] = 10

with open(caminho_schema, "w", encoding="utf-8") as arquivo:
    json.dump(schema, arquivo, indent=4, ensure_ascii=False)

print("Schema atualizado com sucesso!")
print("Categorias:", len(schema["categorias_observadas"]))
print("Regras numéricas:", len(schema["regras_numericas"]))

Schema atualizado com sucesso!
Categorias: 4
Regras numéricas: 7


In [64]:
with open(caminho_schema, encoding="utf-8") as arquivo:
    schema_atualizado = json.load(arquivo)

print("Versão:", schema_atualizado["versao"])

print("\nCategorias observadas:")
for coluna, valores in schema_atualizado["categorias_observadas"].items():
    print(f"{coluna}: {valores}")

print("\nRegras numéricas:")
for coluna, regras in schema_atualizado["regras_numericas"].items():
    print(f"{coluna}: {regras}")

Versão: 1.0.0

Categorias observadas:
faixa_etaria_mae: ['10–14', '15–19', '20–24', '25–29', '30–34', '35–39', '40–44', '45–49', '>=50', 'ignorado']
estado_civil_mae: ['1.0', '2.0', '3.0', '4.0', '5.0', 'ignorado']
escolaridade_mae: ['0.0', '1.0', '2.0', '3.0', '4.0', '5.0', 'ignorado']
tipo_gravidez: ['1.0', '2.0', '3.0', 'ignorado']

Regras numéricas:
qtd_gestacoes: {'minimo': 0, 'inteiro': True, 'permite_nulo': True}
qtd_partos_normais: {'minimo': 0, 'inteiro': True, 'permite_nulo': True}
qtd_partos_cesareas: {'minimo': 0, 'inteiro': True, 'permite_nulo': True}
qtd_filhos_vivos: {'minimo': 0, 'inteiro': True, 'permite_nulo': True}
qtd_filhos_mortos: {'minimo': 0, 'inteiro': True, 'permite_nulo': True}
qtd_consultas_prenatal: {'minimo': 0, 'inteiro': True, 'permite_nulo': True}
mes_inicio_prenatal: {'minimo': 1, 'inteiro': True, 'permite_nulo': True, 'maximo': 10}


In [65]:
import importlib
import src.ml.predict as modulo_predict

importlib.reload(modulo_predict)

predict_one = modulo_predict.predict_one

# Teste de entrada válida
resultado = predict_one(dados_teste)
print("TESTE 1 - Entrada válida:", resultado)

# Testes de entradas inválidas
casos_invalidos = [
    ("Gestação negativa", "qtd_gestacoes", -5),
    ("Mês inválido", "mes_inicio_prenatal", 30),
    ("Gestação decimal", "qtd_gestacoes", 1.5),
    ("Categoria desconhecida", "tipo_gravidez", "categoria_inexistente")
]

for descricao, campo, valor in casos_invalidos:
    dados = dados_teste.copy()
    dados[campo] = valor

    try:
        predict_one(dados)
        print(f"{descricao}: FALHOU - entrada aceita")
    except ValueError as erro:
        print(f"{descricao}: OK - {erro}")

TESTE 1 - Entrada válida: {'versao_modelo': '1.0.0', 'score_baixo_peso': 0.428166, 'classificacao': None, 'limiar': None, 'calibrado': False}
Gestação negativa: OK - qtd_gestacoes deve ser maior ou igual a 0.
Mês inválido: OK - mes_inicio_prenatal deve ser menor ou igual a 10.
Gestação decimal: OK - qtd_gestacoes deve ser inteiro.
Categoria desconhecida: OK - Categoria inválida para tipo_gravidez: categoria_inexistente. Valores aceitos: ['1.0', '2.0', '3.0', 'ignorado']


In [66]:
# TESTE 6: Categorias enviadas como números inteiros

dados_numericos = dados_teste.copy()

dados_numericos["estado_civil_mae"] = 1
dados_numericos["escolaridade_mae"] = 3
dados_numericos["tipo_gravidez"] = 1

try:
    resultado = predict_one(dados_numericos)
    print("TESTE 6 - OK:", resultado)
except Exception as erro:
    print("TESTE 6 - ERRO:", erro)


# TESTE 7: Código ignorado enviado como número 9

dados_ignorados = dados_teste.copy()
dados_ignorados["estado_civil_mae"] = 9

try:
    resultado = predict_one(dados_ignorados)
    print("TESTE 7 - OK:", resultado)
except Exception as erro:
    print("TESTE 7 - ERRO:", erro)

TESTE 6 - ERRO: Invalid value '1.0' for dtype 'int64'
TESTE 7 - ERRO: Categoria inválida para estado_civil_mae: 9.0. Valores aceitos: ['1.0', '2.0', '3.0', '4.0', '5.0', 'ignorado']


In [67]:
import importlib
import src.ml.predict as modulo_predict

importlib.reload(modulo_predict)
predict_one = modulo_predict.predict_one

# TESTE 6
dados_numericos = dados_teste.copy()
dados_numericos["estado_civil_mae"] = 1
dados_numericos["escolaridade_mae"] = 3
dados_numericos["tipo_gravidez"] = 1

try:
    resultado = predict_one(dados_numericos)
    print("TESTE 6 - OK:", resultado)
except Exception as erro:
    print("TESTE 6 - ERRO:", erro)


# TESTE 7
dados_ignorados = dados_teste.copy()
dados_ignorados["estado_civil_mae"] = 9

try:
    resultado = predict_one(dados_ignorados)
    print("TESTE 7 - OK:", resultado)
except Exception as erro:
    print("TESTE 7 - ERRO:", erro)

TESTE 6 - OK: {'versao_modelo': '1.0.0', 'score_baixo_peso': 0.430772, 'classificacao': None, 'limiar': None, 'calibrado': False}
TESTE 7 - OK: {'versao_modelo': '1.0.0', 'score_baixo_peso': 0.585542, 'classificacao': None, 'limiar': None, 'calibrado': False}


In [68]:
# Testa faixa etaria não observada no treinamento
dados_idade_nova = dados_teste.copy()
dados_idade_nova["faixa_etaria_mae"] = "<=9"

try:
    resultado = predict_one(dados_idade_nova)
    print("TESTE 8 - ACEITO:", resultado)
except ValueError as erro:
    print("TESTE 8 - REJEITADO:", erro)

TESTE 8 - REJEITADO: Categoria inválida para faixa_etaria_mae: <=9. Valores aceitos: ['10–14', '15–19', '20–24', '25–29', '30–34', '35–39', '40–44', '45–49', '>=50', 'ignorado']
